In [1]:
import json
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import re

In [2]:
# Configuration
DATASET_PATH = "/kaggle/input/llm-training-in-multilingual-languages"
MAX_PROMPT_TOKENS = 200
MAX_TARGET_TOKENS = 300
MIN_RESPONSE_LENGTH = 20

In [3]:
def load_jsonl_safe(filepath):
    """Load JSONL with error handling"""
    records = []
    bad_lines = 0

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                bad_lines += 1
                continue

    print(f"{filepath} → loaded {len(records)} rows, skipped {bad_lines} bad lines")
    return pd.DataFrame(records)

In [4]:
def clean_text(text):
    """Clean and normalize text"""
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = " ".join(text.split())
    return text.strip()

def estimate_tokens(text):
    """Rough token estimation (split on whitespace)"""
    return len(text.split())

def check_language_script(text, language):
    """Verify text is in expected script"""
    if language == "si":
        # Sinhala unicode range: 0D80-0DFF
        return bool(re.search(r'[\u0D80-\u0DFF]', text))
    elif language == "ta":
        # Tamil unicode range: 0B80-0BFF
        return bool(re.search(r'[\u0B80-\u0BFF]', text))
    elif language == "en":
        # English: basic latin
        return bool(re.search(r'[a-zA-Z]', text))
    return True

In [5]:
# Load datasets
print("Loading datasets...")
df_en = load_jsonl_safe(f"{DATASET_PATH}/english_dataset.jsonl")
df_ta = load_jsonl_safe(f"{DATASET_PATH}/tamil_dataset.jsonl")
df_si = load_jsonl_safe(f"{DATASET_PATH}/sinhala_dataset.jsonl")

Loading datasets...
/kaggle/input/llm-training-in-multilingual-languages/english_dataset.jsonl → loaded 1000 rows, skipped 0 bad lines
/kaggle/input/llm-training-in-multilingual-languages/tamil_dataset.jsonl → loaded 998 rows, skipped 2 bad lines
/kaggle/input/llm-training-in-multilingual-languages/sinhala_dataset.jsonl → loaded 996 rows, skipped 4 bad lines


In [6]:
# Add language tags
df_en["language"] = "en"
df_si["language"] = "si"
df_ta["language"] = "ta"

In [7]:
# Combine datasets
df = pd.concat([df_en, df_si, df_ta], ignore_index=True)
print(f"\nInitial dataset size: {len(df)} rows")


Initial dataset size: 2994 rows


In [8]:
# Remove rows with missing values
df = df.dropna(subset=["user_emotion", "user_input", "bot_response"])
print(f"After removing NaN: {len(df)} rows")

# Clean text
print("\nCleaning text...")
df["user_input"] = df["user_input"].apply(clean_text)
df["bot_response"] = df["bot_response"].apply(clean_text)
df["user_emotion"] = df["user_emotion"].str.lower().str.strip()

# Remove very short inputs/responses
df = df[df["user_input"].str.len() > 5]
df = df[df["bot_response"].str.len() > MIN_RESPONSE_LENGTH]
print(f"After length filter: {len(df)} rows")

# Remove duplicates
df = df.drop_duplicates(subset=["user_input", "bot_response"])
print(f"After removing duplicates: {len(df)} rows")

After removing NaN: 2994 rows

Cleaning text...
After length filter: 2994 rows
After removing duplicates: 2979 rows


In [9]:
# Verify language scripts
print("\nVerifying language scripts...")
df["valid_script"] = df.apply(
    lambda row: check_language_script(row["user_input"], row["language"]) and 
                check_language_script(row["bot_response"], row["language"]), 
    axis=1
)
df = df[df["valid_script"]].drop(columns=["valid_script"])
print(f"After script validation: {len(df)} rows")


Verifying language scripts...
After script validation: 2979 rows


In [10]:
# Create prompts in standard format
df["prompt"] = (
    "<|system|>\nYou are an empathetic assistant. The user is feeling " + 
    df["user_emotion"] + ".\n" +
    "<|user|>\n" + df["user_input"] + "\n" +
    "<|assistant|>\n"
)

df["target"] = df["bot_response"]

In [11]:
# Estimate tokens
df["prompt_tokens"] = df["prompt"].apply(estimate_tokens)
df["target_tokens"] = df["target"].apply(estimate_tokens)

# Filter by token length
df = df[
    (df["prompt_tokens"] < MAX_PROMPT_TOKENS) & 
    (df["target_tokens"] < MAX_TARGET_TOKENS)
]
print(f"After token filtering: {len(df)} rows")

After token filtering: 2979 rows


In [12]:
# Show statistics
print("\n=== Dataset Statistics ===")
print(f"\nEmotion distribution:")
print(df["user_emotion"].value_counts())
print(f"\nLanguage distribution:")
print(df["language"].value_counts())
print(f"\nPrompt token stats:")
print(df["prompt_tokens"].describe())
print(f"\nTarget token stats:")
print(df["target_tokens"].describe())


=== Dataset Statistics ===

Emotion distribution:
user_emotion
happy        429
surprised    428
sad          427
angry        426
fearful      426
neutral      426
disgust      417
Name: count, dtype: int64

Language distribution:
language
en    999
si    995
ta    985
Name: count, dtype: int64

Prompt token stats:
count    2979.000000
mean       20.050017
std         2.040519
min        16.000000
25%        19.000000
50%        20.000000
75%        21.000000
max        28.000000
Name: prompt_tokens, dtype: float64

Target token stats:
count    2979.000000
mean       16.559248
std         5.174792
min         4.000000
25%        13.000000
50%        15.000000
75%        19.000000
max        44.000000
Name: target_tokens, dtype: float64


In [13]:
# Create stratified splits
print("\n=== Creating train/val/test splits ===")

# Create stratification column (language + emotion)
df["strat_col"] = df["language"] + "_" + df["user_emotion"]

# Split: 80% train, 10% val, 10% test
train_df, temp_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42,
    stratify=df["strat_col"]
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    random_state=42,
    stratify=temp_df["strat_col"]
)

print(f"Train: {len(train_df)} rows")
print(f"Val: {len(val_df)} rows")
print(f"Test: {len(test_df)} rows")


=== Creating train/val/test splits ===
Train: 2383 rows
Val: 298 rows
Test: 298 rows


In [14]:
# Save datasets
print("\nSaving datasets...")
train_df[["prompt", "target", "language", "user_emotion"]].to_json(
    "train.jsonl", orient="records", lines=True, force_ascii=False
)

val_df[["prompt", "target", "language", "user_emotion"]].to_json(
    "val.jsonl", orient="records", lines=True, force_ascii=False
)

test_df[["prompt", "target", "language", "user_emotion"]].to_json(
    "test.jsonl", orient="records", lines=True, force_ascii=False
)

# Save full processed dataset for reference
df[["prompt", "target", "language", "user_emotion", "prompt_tokens", "target_tokens"]].to_json(
    "full_processed.jsonl", orient="records", lines=True, force_ascii=False
)
print("\n✓ Preprocessing complete!")
print("Files saved: train.jsonl, val.jsonl, test.jsonl, full_processed.jsonl")


Saving datasets...

✓ Preprocessing complete!
Files saved: train.jsonl, val.jsonl, test.jsonl, full_processed.jsonl
